# config

> Shapes for the wav2vec 2.0 base checkpoint.
> 
> The model is a convolutional feature encoder, a feature projection, a
> convolutional position embedding, and a 12-layer transformer. A linear
> head maps each 20 ms frame to a character for CTC. These defaults match
> `facebook/wav2vec2-base-960h`.


In [ ]:
#| default_exp config


In [ ]:
#| hide
from nbdev.showdoc import *


The checkpoint is the 960-hour LibriSpeech fine-tune of wav2vec 2.0 base. Audio is 16 kHz mono. The seven strided convolutions downsample by 320, so one frame covers 20 ms.


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
from dataclasses import dataclass


In [ ]:
#| export
@dataclass(frozen=True)
class Wav2Vec2Config:
    """Wav2vec 2.0 base with a CTC head. `conv_*` describe the feature encoder."""

    vocab_size: int = 32
    hidden_size: int = 768
    num_hidden_layers: int = 12
    num_attention_heads: int = 12
    intermediate_size: int = 3072
    layer_norm_eps: float = 1e-5
    group_norm_eps: float = 1e-5
    conv_dim: tuple[int, ...] = (512,) * 7
    conv_kernel: tuple[int, ...] = (10, 3, 3, 3, 3, 2, 2)
    conv_stride: tuple[int, ...] = (5, 2, 2, 2, 2, 2, 2)
    conv_bias: bool = False
    num_conv_pos_embeddings: int = 128
    num_conv_pos_embedding_groups: int = 16
    pad_token_id: int = 0
    sample_rate: int = 16_000

    def __post_init__(self) -> None:
        if not len(self.conv_dim) == len(self.conv_kernel) == len(self.conv_stride):
            raise ValueError("conv_dim, conv_kernel and conv_stride need the same length")
        if self.hidden_size % self.num_attention_heads:
            raise ValueError(f"{self.hidden_size} is not divisible by {self.num_attention_heads} heads")
        if self.hidden_size % self.num_conv_pos_embedding_groups:
            raise ValueError(f"{self.hidden_size} is not divisible by {self.num_conv_pos_embedding_groups} groups")

    @property
    def feature_size(self) -> int:
        """Channel count out of the feature encoder, before the projection."""
        return self.conv_dim[-1]

    @property
    def head_dim(self) -> int:
        return self.hidden_size // self.num_attention_heads

    @property
    def samples_per_frame(self) -> int:
        """Hop of the feature encoder: 320 samples, or 20 ms, for the base model."""
        hop = 1
        for stride in self.conv_stride:
            hop *= stride
        return hop

    def output_length(self, num_samples: int) -> int:
        """Frames the feature encoder produces for `num_samples` of audio."""
        length = num_samples
        for kernel, stride in zip(self.conv_kernel, self.conv_stride, strict=True):
            length = (length - kernel) // stride + 1
        if length <= 0:
            raise ValueError(f"{num_samples} samples are too short for the feature encoder")
        return length


In [ ]:
#| export
def base_960h_config() -> Wav2Vec2Config:
    """Config of `facebook/wav2vec2-base-960h`."""
    return Wav2Vec2Config()


In [ ]:
#| export
def tiny_config() -> Wav2Vec2Config:
    """A few small blocks of the same wiring, for graph checks and parity tests."""
    return Wav2Vec2Config(
        vocab_size=12,
        hidden_size=32,
        num_hidden_layers=2,
        num_attention_heads=4,
        intermediate_size=64,
        conv_dim=(16, 16, 16),
        conv_kernel=(4, 3, 2),
        conv_stride=(2, 2, 2),
        num_conv_pos_embeddings=8,
        num_conv_pos_embedding_groups=4,
    )


One second of audio is 49 frames, and the tiny config turns 400 samples into 49 too.


In [ ]:
assert base_960h_config().samples_per_frame == 320
assert base_960h_config().output_length(16_000) == 49
assert tiny_config().output_length(400) == 49
